# Clustering Track — Credit Card Customer Segmentation
**23CSE301 Machine Learning Capstone — Review 2**

Pipeline: load & audit → EDA → cleaning & scaling → **K-Means** (Elbow) → **Agglomerative Hierarchical** (Dendrogram, linkage comparison) → Silhouette / Davies-Bouldin / Calinski-Harabasz → PCA & t-SNE visualisation → cluster profiling.

Clustering is unsupervised: there is no target column, and nothing here is evaluated against labels.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score)
from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42
sns.set_style('whitegrid')
sns.set_palette('colorblind')
plt.rcParams['figure.figsize'] = (8, 5)

## 1. Dataset loading & audit

In [ ]:
DATA_PATH = "../data/CC GENERAL.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Missing values:\n", missing if len(missing) else "None")
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate CUST_IDs:", df['CUST_ID'].duplicated().sum())

## 2. EDA

In [ ]:
features = [col for col in df.columns if col != 'CUST_ID']

fig, axes = plt.subplots(5, 4, figsize=(20, 18))
axes = axes.flatten()
for i, col in enumerate(features):
    sns.histplot(df[col].dropna(), bins=40, kde=True, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}', fontsize=9)
for j in range(len(features), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(14, 11))
sns.heatmap(df[features].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True, annot_kws={'size': 7})
plt.title('Correlation Heatmap — Credit Card Features')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x='PURCHASES', y='CREDIT_LIMIT', alpha=0.4, ax=axes[0])
axes[0].set_title('Purchases vs Credit Limit')
sns.scatterplot(data=df, x='BALANCE', y='CASH_ADVANCE', alpha=0.4, ax=axes[1])
axes[1].set_title('Balance vs Cash Advance')
plt.tight_layout(); plt.show()

> **Observations (write your own):** note the heavy right-skew in monetary columns, the strong correlations (e.g. PURCHASES vs ONEOFF_PURCHASES, PURCHASES_FREQUENCY vs PURCHASES_INSTALLMENTS_FREQUENCY), and what these imply for distance-based clustering.

## 3. Preprocessing
- Drop `CUST_ID` (identifier, no information).
- Impute missing `MINIMUM_PAYMENTS` and `CREDIT_LIMIT` with the **median** (columns are heavily skewed, so the mean would be pulled by outliers).
- Apply `log1p` to the skewed monetary / count columns so that a few extreme customers do not dominate Euclidean distances. Frequency columns (0–1) and TENURE are left as is.
- Standardise all features (`StandardScaler`) — K-Means and Ward linkage are both distance based. There is no train/test split in unsupervised clustering, so scaling on the full dataset is not leakage.

In [ ]:
data = df.drop(columns=['CUST_ID']).copy()
df_before = data.copy()                       # keep original-scale copy for profiling

imputer = SimpleImputer(strategy='median')
data[:] = imputer.fit_transform(data)

skewed_cols = ['BALANCE', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE',
               'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS']
print("Skewness before log1p:\n", data[skewed_cols].skew().round(2).to_string())
data[skewed_cols] = np.log1p(data[skewed_cols])
print("\nSkewness after log1p:\n", data[skewed_cols].skew().round(2).to_string())

### Feature engineering

In [ ]:
# Engineered feature: payment-to-balance behaviour. A customer who pays far more than the
# balance they carry behaves very differently from one who revolves a balance.
raw = df_before.fillna(df_before.median())
data['PAYMENT_TO_BALANCE'] = np.log1p(raw['PAYMENTS'] / (raw['BALANCE'] + 1))

scaler = StandardScaler()
X = scaler.fit_transform(data)
X_df = pd.DataFrame(X, columns=data.columns)
print("Final feature matrix:", X.shape)

In [ ]:
# Helper to compute the 3 mandatory internal metrics
def cluster_metrics(X, labels):
    return {'Silhouette': silhouette_score(X, labels),
            'Davies-Bouldin': davies_bouldin_score(X, labels),
            'Calinski-Harabasz': calinski_harabasz_score(X, labels)}

## 4. K-Means Clustering
### 4.1 Elbow curve (and silhouette) to choose k

In [ ]:
K_RANGE = range(2, 11)
inertias, sil_scores = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, km.labels_))
    
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_RANGE), inertias, 'o-')
axes[0].set_title('Elbow Curve — K-Means'); axes[0].set_xlabel('Number of clusters (k)'); axes[0].set_ylabel('Inertia')
axes[1].plot(list(K_RANGE), sil_scores, 'o-', color='tab:orange')
axes[1].set_title('Silhouette Score vs k'); axes[1].set_xlabel('Number of clusters (k)'); axes[1].set_ylabel('Silhouette score')
plt.tight_layout(); plt.show()
pd.DataFrame({'k': list(K_RANGE), 'Inertia': inertias, 'Silhouette': sil_scores})

> **Choosing k (write your own justification):** point to where the elbow bends and what the silhouette says, then set `K_CHOSEN` below.

In [ ]:
K_CHOSEN = 4        # <-- set from the elbow / silhouette plots above
kmeans = KMeans(n_clusters=K_CHOSEN, n_init=10, random_state=RANDOM_STATE).fit(X)
km_labels = kmeans.labels_
print("K-Means cluster sizes:\n", pd.Series(km_labels).value_counts().sort_index().to_string())

## 5. Agglomerative Hierarchical Clustering
### 5.1 Dendrogram

In [ ]:
Z = linkage(X, method='ward')
plt.figure(figsize=(13, 6))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=9, show_contracted=True)
plt.title('Dendrogram — Ward Linkage (truncated to last 30 merges)')
plt.xlabel('Cluster (size in brackets)'); plt.ylabel('Ward distance')
plt.tight_layout(); plt.show()

### 5.2 Linkage strategy comparison

In [ ]:
linkage_rows = {}
for link in ['ward', 'complete', 'average', 'single']:
    lab = AgglomerativeClustering(n_clusters=K_CHOSEN, linkage=link).fit_predict(X)
    linkage_rows[link] = cluster_metrics(X, lab)
linkage_df = pd.DataFrame(linkage_rows).T
display(linkage_df.style.format('{:.4f}'))
print("Smallest cluster size per linkage:")
for link in linkage_rows:
    sizes = pd.Series(AgglomerativeClustering(n_clusters=K_CHOSEN, linkage=link).fit_predict(X)).value_counts()
    print(f"  {link:9s}", sizes.to_dict())

In [ ]:
BEST_LINKAGE = 'ward'    # <-- choose using the table and cluster sizes above
agg = AgglomerativeClustering(n_clusters=K_CHOSEN, linkage=BEST_LINKAGE)
agg_labels = agg.fit_predict(X)
print("Agglomerative cluster sizes:\n", pd.Series(agg_labels).value_counts().sort_index().to_string())

## 6. Evaluation — Silhouette, Davies-Bouldin, Calinski-Harabasz
Higher is better for Silhouette and Calinski-Harabasz; **lower** is better for Davies-Bouldin.

In [ ]:
eval_df = pd.DataFrame({
    'K-Means': cluster_metrics(X, km_labels),
    f'Agglomerative ({BEST_LINKAGE})': cluster_metrics(X, agg_labels),
}).T
eval_df['n_clusters'] = [K_CHOSEN, K_CHOSEN]
display(eval_df.style.format({'Silhouette': '{:.4f}', 'Davies-Bouldin': '{:.4f}', 'Calinski-Harabasz': '{:.1f}'}))
print("Agreement between the two clusterings (Adjusted Rand Index): %.4f" % adjusted_rand_score(km_labels, agg_labels))